# 10 -Evolução do baseline (LightGBM):  preços quebrados (P3)
Mede se o alerta do baseline evolui quando saímos dos erros extremos dos rótulos e passamos a testar erros moderados, e corrige a MAD, que no baseline foi calculada em linhas que o modelo já tinha visto.

**Entradas:** tabelas `squad1/ml/features_precos_observacoes` (292.000 linhas) e `squad1/ml/features_precos_sku` (3.805 linhas), geradas no notebook 08. O modelo base é o mesmo do notebook 09 (7 colunas, 68 árvores).

**Saídas:** recall do alerta em erros sintéticos de 2 a 8 vezes, taxa de falso alarme nas linhas normais e a comparação entre a versão 1 (baseline) e a versão 2 (MAD fora do treino).

**Como rodar:** use "Executar tudo", sempre na ordem. A célula 2 instala o lightgbm (%pip) e precisa ficar logo no começo, antes de qualquer outra.

In [0]:
%pip install lightgbm python-dotenv

In [0]:
# ------------------------------------------------------------
# Configuração
# Importações, acesso ao lake (ADLS) e a função de leitura de tabelas Delta.
# Esta célula não lê nem grava dados.
# ------------------------------------------------------------
import os
from dotenv import load_dotenv
from pyspark.sql import functions as F

# Lê o arquivo .env (mesma pasta do notebook), onde ficam as credenciais do ADLS.
load_dotenv(".env")

# Endereço do lake: conta de armazenamento e pasta raiz da Squad 1.
STORAGE_ACCOUNT = "internshipdatalake"
lakehouse_base = f"abfss://squad1@{STORAGE_ACCOUNT}.dfs.core.windows.net"

# Credenciais de acesso ao ADLS (autenticação OAuth com um aplicativo do Azure).
# No Serverless o spark.conf.set de credenciais é bloqueado,
# por isso elas vão em cada leitura, pelo .options(**adls_options).
adls_options = {
    "fs.azure.account.auth.type": "OAuth",
    "fs.azure.account.oauth.provider.type": "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider",
    "fs.azure.account.oauth2.client.id": os.getenv("ADLS_CLIENT_ID"),
    "fs.azure.account.oauth2.client.secret": os.getenv("ADLS_CLIENT_SECRET"),
    "fs.azure.account.oauth2.client.endpoint": f"https://login.microsoftonline.com/{os.getenv('ADLS_TENANT_ID')}/oauth2/token",
}

def ler_delta_spark(caminho):
    """Lê uma tabela Delta do ADLS.

    caminho: endereço abfss:// da tabela (ex.: .../ml/features_precos_sku).
    Retorna um DataFrame do Spark.
    """
    return spark.read.format("delta").options(**adls_options).load(caminho)

In [0]:
# ------------------------------------------------------------
# Carga dos dados e separação treino / teste
# Lê as duas tabelas de features geradas no notebook 08, traz as observações
# para o pandas e separa treino (folds 1 a 4) e teste (fold 0), como no notebook 09.
# Também define as colunas do modelo base e a função de erro (MAE).
# Não altera nada no lake: só lê.
# ------------------------------------------------------------
import numpy as np
import pandas as pd
import lightgbm as lgb

obs_path = f"{lakehouse_base}/ml/features_precos_observacoes"  # uma linha por observação
sku_path = f"{lakehouse_base}/ml/features_precos_sku"          # uma linha por SKU

df_obs = ler_delta_spark(obs_path)
df_sku = ler_delta_spark(sku_path)
print("features_precos_observacoes:", df_obs.count(), "linhas,", len(df_obs.columns), "colunas")
print("features_precos_sku:", df_sku.count(), "linhas,", len(df_sku.columns), "colunas")

# Colunas do modelo base (as mesmas 7 do notebook 09). As 3 primeiras são categóricas.
FEATURES_BASE = ["id_categoria", "id_categoria_pai", "unidade_cod",
                 "tamanho", "log_tamanho", "tamanho_relativo", "tamanho_rank_pct"]
CATEGORICAS_BASE = ["id_categoria", "id_categoria_pai", "unidade_cod"]
N_ARVORES_BASE = 68   # achado na parada antecipada do notebook 09

pdf = df_obs.toPandas()   # 292.000 linhas cabem tranquilamente na memória

# Treino: folds 1 a 4, só linhas "normais" (usar_no_treino == 1).
treino = pdf[(pdf["conjunto"] == "treino") & (pdf["usar_no_treino"] == 1)].copy()
# Teste: TODAS as linhas do fold 0 (normais e quebradas), porque o alerta é medido nelas.
teste_todas = pdf[pdf["conjunto"] == "teste"].copy()
# Teste normal: só as linhas normais do fold 0, usadas para o MAE.
teste_normais = teste_todas[teste_todas["usar_no_treino"] == 1]

def mae(real, previsto):
    """Erro absoluto médio (em R$): média do tamanho do erro, sem sinal."""
    return float(np.mean(np.abs(real - previsto)))

print("\nLinhas de treino (normais):", len(treino))
print("Linhas de teste (todas):", len(teste_todas), "| normais:", len(teste_normais))
print("SKUs no teste:", teste_todas["sku"].nunique())
print("Rótulos no teste: fraco", int(teste_todas["rotulo_fraco"].sum()),
      "| apoio", int(teste_todas["rotulo_apoio"].sum()))

In [0]:
# ------------------------------------------------------------
# Versão 1 (baseline): modelo base + alerta com MAD calculada no treino
# Treina o LightGBM de 7 colunas (68 árvores), mede o MAE nas linhas normais
# do teste e refaz o alerta do notebook 09 (k = 5). Serve de ponto de comparação
# para as melhorias das próximas etapas.
# Não altera nada no lake.
# ------------------------------------------------------------
# 1) Modelo base: igual ao do notebook 09.
modelo_base = lgb.LGBMRegressor(
    objective="regression_l1", n_estimators=N_ARVORES_BASE, learning_rate=0.05,
    num_leaves=31, min_child_samples=100, random_state=42, verbose=-1,
)
modelo_base.fit(treino[FEATURES_BASE], treino["preco"], categorical_feature=CATEGORICAS_BASE)

# Preço esperado de cada linha do treino e do teste.
treino["previsto"] = modelo_base.predict(treino[FEATURES_BASE])
teste_todas["previsto"] = modelo_base.predict(teste_todas[FEATURES_BASE])
teste_normais = teste_todas[teste_todas["usar_no_treino"] == 1]   # refaz, agora com "previsto"

print("MAE nas linhas normais do teste: R$", round(mae(teste_normais["preco"], teste_normais["previsto"]), 2),
      "(esperado 9,45)")

# 2) Score e z: mesmas contas do notebook 09.
def calcular_score(df):
    """Score = log(preço real / preço esperado). Preço <= 0 vira 0,01 para o log existir."""
    return np.log(np.maximum(df["preco"], 0.01) / df["previsto"])

def calcular_parametros(df_ref):
    """Centro e MAD (bagunça típica) do score, globais e por categoria raiz.

    df_ref: linhas normais usadas como referência (precisa ter 'previsto').
    Retorna um dicionário com os 4 parâmetros usados por calcular_z.
    """
    score = calcular_score(df_ref)
    centro_global = score.median()
    mad_global = (score - centro_global).abs().median()
    por_raiz = score.groupby(df_ref["id_categoria_pai"])
    return {
        "centro_global": centro_global,
        "mad_global": mad_global,
        "centro_raiz": por_raiz.median(),
        "mad_raiz": por_raiz.apply(lambda s: (s - s.median()).abs().median()),
    }

def calcular_z(df, p):
    """Distância do score ao centro do grupo, em unidades de MAD (z). p = parâmetros."""
    centro = df["id_categoria_pai"].map(p["centro_raiz"]).fillna(p["centro_global"])
    mad = df["id_categoria_pai"].map(p["mad_raiz"]).fillna(p["mad_global"])
    return (calcular_score(df) - centro) / (1.4826 * mad)

def avaliar_alertas(df, k):
    """Precision, recall e taxa de alerta (em %) para o limite k."""
    alerta = df["z"].abs() > k
    n_alertas = int(alerta.sum())
    saida = {"k": k, "n_alertas": n_alertas, "taxa_alerta_%": alerta.mean() * 100}
    for rotulo in ["fraco", "apoio"]:
        real = df["rotulo_" + rotulo] == 1
        acertos = int((alerta & real).sum())
        saida["precision_" + rotulo] = acertos / n_alertas * 100 if n_alertas > 0 else np.nan
        saida["recall_" + rotulo] = acertos / int(real.sum()) * 100
    return saida

# 3) Versão 1: MAD calculada nas linhas normais do treino (que o modelo já viu).
params_v1 = calcular_parametros(treino)
print("\nMAD global (v1):", round(params_v1["mad_global"], 3), "(esperado 0,336)")

teste_todas["z_v1"] = calcular_z(teste_todas, params_v1)
tabela_v1 = pd.DataFrame([avaliar_alertas(teste_todas.assign(z=teste_todas["z_v1"]), k)
                          for k in [3, 4, 5, 6, 8, 10]])
print("\n=== Versão 1 no teste (todas as linhas) ===")
print(tabela_v1.round(1).to_string(index=False))

In [0]:
# ------------------------------------------------------------
# Erros sintéticos moderados
# Pega as linhas NORMAIS do teste e multiplica o preço de todas por um fator
# (2, 3, 5, 8 ou o inverso), simulando um erro de preço de tamanho conhecido.
# O preço esperado (previsto) não muda, pois o preço não é entrada do modelo.
# Um cenário por fator, mais o original (x1) para medir falsos alarmes.
# Não altera nada no lake.
# ------------------------------------------------------------
FATORES = {
    "÷5": 1 / 5, "÷3": 1 / 3, "÷2": 1 / 2,
    "x1 (original)": 1,
    "x2": 2, "x3": 3, "x5": 5, "x8": 8,
}

def injetar_erro(df, fator):
    """Cópia de df com o preço multiplicado por 'fator'.

    df: linhas normais do teste (já com 'previsto').
    fator: 2 = preço dobrado, 0,5 = preço cortado pela metade, 1 = sem erro.
    """
    saida = df.copy()
    saida["preco"] = saida["preco"] * fator
    saida["fator"] = fator
    return saida

# Conferências antes de injetar: as linhas normais não podem ter rótulo nem preço <= 0.
print("Linhas normais do teste:", len(teste_normais))
print("Soma dos rótulos nessas linhas (esperado 0):",
      int(teste_normais["rotulo_fraco"].sum() + teste_normais["rotulo_apoio"].sum()))
print("Menor preço (esperado > 0): R$", round(teste_normais["preco"].min(), 2))

# Um DataFrame por cenário, já com o z da versão 1.
sinteticos = {}
for nome, fator in FATORES.items():
    df_cenario = injetar_erro(teste_normais, fator)
    df_cenario["z"] = calcular_z(df_cenario, params_v1)
    sinteticos[nome] = df_cenario

# Resumo para conferir se os cenários fazem sentido.
resumo = pd.DataFrame([{
    "cenário": nome,
    "linhas": len(df_c),
    "razão mediana (real/esperado)": float((df_c["preco"] / df_c["previsto"]).median()),
    "z mediano": float(df_c["z"].median()),
} for nome, df_c in sinteticos.items()])
print("\n=== Cenários sintéticos ===")
print(resumo.round(2).to_string(index=False))

In [0]:
# ------------------------------------------------------------
# Recall do alerta nos erros sintéticos (versão 1)
# Para cada cenário e cada limite k, calcula a % de linhas com |z| > k.
#   - Nos cenários com erro (÷5 a x8), todas as linhas são erros: a % é o RECALL.
#   - No cenário original (x1), nenhuma linha é erro: a % é a taxa de FALSO ALARME.
# Não altera nada no lake.
# ------------------------------------------------------------
KS_SINTETICO = [2, 3, 4, 5, 6]

tabela_sint_v1 = pd.DataFrame({
    f"k={k}": {nome: (df_c["z"].abs() > k).mean() * 100 for nome, df_c in sinteticos.items()}
    for k in KS_SINTETICO
})
tabela_sint_v1.index.name = "cenário"

print("=== Versão 1: % de linhas em alerta por cenário e limite k ===")
print("(erros: leia como recall | x1 (original): leia como falso alarme)\n")
print(tabela_sint_v1.round(1).to_string())

# Quantas linhas normais (sem erro) entram em alerta, em número absoluto.
original = sinteticos["x1 (original)"]
print("\n=== Falsos alarmes no original (x1): nº de linhas de", len(original), "===")
for k in KS_SINTETICO:
    print(f"k={k}:", int((original['z'].abs() > k).sum()))

In [0]:
# ------------------------------------------------------------
# MAD fora do treino (previsões out-of-fold)
# Para cada fold do treino (1 a 4), treina o modelo base nos outros 3 folds e
# prevê o fold deixado de fora. Como a divisão é por SKU, cada linha do treino
# recebe um preço esperado de um modelo que nunca viu o SKU dela.
# Com essas previsões, recalcula o centro e a MAD (versão 2) e compara com a v1.
# Não altera nada no lake.
# ------------------------------------------------------------
folds_treino = sorted(treino["fold"].unique())
print("Folds do treino:", folds_treino)

treino["previsto_oof"] = np.nan
for f in folds_treino:
    fora = treino["fold"] == f   # linhas do fold deixado de fora
    m = lgb.LGBMRegressor(
        objective="regression_l1", n_estimators=N_ARVORES_BASE, learning_rate=0.05,
        num_leaves=31, min_child_samples=100, random_state=42, verbose=-1,
    )
    m.fit(treino.loc[~fora, FEATURES_BASE], treino.loc[~fora, "preco"],
          categorical_feature=CATEGORICAS_BASE)
    treino.loc[fora, "previsto_oof"] = m.predict(treino.loc[fora, FEATURES_BASE])
    print(f"Fold {f} fora: {int(fora.sum())} linhas | MAE R$",
          round(mae(treino.loc[fora, "preco"], treino.loc[fora, "previsto_oof"]), 2))

print("\nNulos em previsto_oof (esperado 0):", int(treino["previsto_oof"].isna().sum()))
print("MAE no treino com o modelo que já viu as linhas (v1): R$",
      round(mae(treino["preco"], treino["previsto"]), 2))
print("MAE no treino fora do fold (v2):                      R$",
      round(mae(treino["preco"], treino["previsto_oof"]), 2))

# Versão 2: centro e MAD calculados com as previsões fora do fold.
params_v2 = calcular_parametros(treino.assign(previsto=treino["previsto_oof"]))

print("\nMAD global: v1 =", round(params_v1["mad_global"], 3), "| v2 =", round(params_v2["mad_global"], 3))
comparacao_mad = pd.DataFrame({
    "mad_v1": params_v1["mad_raiz"],
    "mad_v2": params_v2["mad_raiz"],
})
comparacao_mad["v2 / v1"] = comparacao_mad["mad_v2"] / comparacao_mad["mad_v1"]
print("\n=== MAD por categoria raiz ===")
print(comparacao_mad.round(3).to_string())

In [0]:
# ------------------------------------------------------------
# Escolha do k com erros sintéticos no TREINO (previsões fora do fold)
# Repete os cenários de erro (÷5 a x8) nas linhas normais do treino, usando o
# preço esperado fora do fold e a MAD da versão 2. A % de linhas em alerta é
# o recall nos cenários com erro e o falso alarme no cenário original (x1).
# O teste NÃO é usado aqui: serve só para confirmar o k escolhido.
# Não altera nada no lake.
# ------------------------------------------------------------
df_oof = treino.assign(previsto=treino["previsto_oof"])   # preço esperado fora do fold

sinteticos_oof = {}
for nome, fator in FATORES.items():
    df_cenario = injetar_erro(df_oof, fator)
    df_cenario["z"] = calcular_z(df_cenario, params_v2)
    sinteticos_oof[nome] = df_cenario

KS_ESCOLHA = [2, 2.5, 3, 3.5, 4, 5]
tabela_oof = pd.DataFrame({
    f"k={k}": {nome: (df_c["z"].abs() > k).mean() * 100 for nome, df_c in sinteticos_oof.items()}
    for k in KS_ESCOLHA
})
tabela_oof.index.name = "cenário"

print("=== Treino fora do fold, MAD v2: % de linhas em alerta ===")
print("(erros: recall | x1 (original): falso alarme)\n")
print(tabela_oof.round(2).to_string())

original_oof = sinteticos_oof["x1 (original)"]
print("\n=== Falsos alarmes no original (x1): nº de linhas de", len(original_oof), "===")
for k in KS_ESCOLHA:
    print(f"k={k}:", int((original_oof['z'].abs() > k).sum()),
          f"({(original_oof['z'].abs() > k).mean() * 100:.3f}%)")

In [0]:
# ------------------------------------------------------------
# Confirmação no teste: versão 1 (k = 5) contra versão 2 (k = 3 e k = 3,5)
# 1) Contra os rótulos reais (fraco e apoio): alertas, precision e recall.
# 2) Nos erros sintéticos do teste: % de linhas em alerta por cenário.
# O k foi escolhido no treino (célula 9); o teste só confirma.
# Não altera nada no lake.
# ------------------------------------------------------------
K_V1 = 5      # baseline
K_V2 = 3      # escolhido pelo critério: menor k com falso alarme <= 0,05% no treino
K_V2_CAUT = 3.5   # alternativa mais conservadora (0 falso alarme no treino)

teste_todas["z_v2"] = calcular_z(teste_todas, params_v2)

versoes = {
    f"v1 (k={K_V1})": ("z_v1", K_V1),
    f"v2 (k={K_V2})": ("z_v2", K_V2),
    f"v2 (k={K_V2_CAUT})": ("z_v2", K_V2_CAUT),
}

# 1) Contra os rótulos reais.
linhas = []
for nome, (coluna_z, k) in versoes.items():
    r = avaliar_alertas(teste_todas.assign(z=teste_todas[coluna_z]), k)
    r["versão"] = nome
    linhas.append(r)
tabela_rotulos = pd.DataFrame(linhas).set_index("versão").drop(columns="k")
print("=== Teste (", len(teste_todas), "linhas): alertas contra os rótulos reais ===")
print(tabela_rotulos.round(1).to_string())

# 2) Nos erros sintéticos do teste (todas as linhas normais com o preço multiplicado).
resultado = {}
for cenario, df_c in sinteticos.items():
    z_v1 = df_c["z"]                       # já calculado com a MAD v1
    z_v2 = calcular_z(df_c, params_v2)     # MAD v2
    resultado[cenario] = {
        f"v1 (k={K_V1})": (z_v1.abs() > K_V1).mean() * 100,
        f"v2 (k={K_V2})": (z_v2.abs() > K_V2).mean() * 100,
        f"v2 (k={K_V2_CAUT})": (z_v2.abs() > K_V2_CAUT).mean() * 100,
    }
tabela_sint_teste = pd.DataFrame(resultado).T
tabela_sint_teste.index.name = "cenário"
print("\n=== Teste, erros sintéticos: % de linhas em alerta ===")
print("(erros: recall | x1 (original): falso alarme)\n")
print(tabela_sint_teste.round(1).to_string())

In [0]:
# ------------------------------------------------------------
# Exemplos concretos
# 1) Alertas extras da versão 2 (k = 3) no teste: linhas em alerta que nenhum
#    rótulo marcou. Compara o preço com o esperado e com a mediana do próprio SKU.
# 2) Cinco exemplos de erro sintético (x5) que a v1 (k = 5) não pegava e a v2 pega.
# Só lê os dados já carregados. Não altera nada no lake.
# ------------------------------------------------------------
# 1) Alertas extras do teste.
extras = teste_todas[
    (teste_todas["z_v2"].abs() > K_V2)
    & (teste_todas["rotulo_fraco"] == 0)
    & (teste_todas["rotulo_apoio"] == 0)
].copy()
extras["razao_esperado"] = extras["preco"] / extras["previsto"]
extras["razao_mediana_sku"] = extras["preco"] / extras["mediana_sku"]
extras = extras.reindex(extras["z_v2"].abs().sort_values(ascending=False).index)

colunas = ["sku", "id_categoria", "tamanho", "preco", "previsto", "mediana_sku",
           "razao_esperado", "razao_mediana_sku", "z_v1", "z_v2"]
print("=== Alertas extras da v2 (k =", K_V2, ") no teste: ", len(extras), "linhas ===")
print(extras[colunas].round(2).to_string(index=False))

# 2) Exemplos sintéticos: x5 que a v1 não pega (|z| <= 5) e a v2 pega (|z| > 3).
df_x5 = sinteticos["x5"].copy()
df_x5["z_v2"] = calcular_z(df_x5, params_v2)
df_x5["preco_original"] = df_x5["preco"] / 5
df_x5["razao_esperado"] = df_x5["preco"] / df_x5["previsto"]
exemplos = df_x5[(df_x5["z"].abs() <= K_V1) & (df_x5["z_v2"].abs() > K_V2)]
print("\nLinhas x5 que a v1 perde e a v2 pega:", len(exemplos), "de", len(df_x5))
print(exemplos.sample(5, random_state=42)[
    ["sku", "id_categoria", "tamanho", "preco_original", "preco", "previsto",
     "razao_esperado", "z", "z_v2"]
].rename(columns={"preco": "preco_com_erro", "z": "z_v1"}).round(2).to_string(index=False))

In [0]:
# ------------------------------------------------------------
# Tabela final: evolução do baseline (v1 contra v2)
# Junta, para o teste: a MAD, o limite k, os alertas, precision e recall contra
# os rótulos reais, o recall nos erros sintéticos e o falso alarme nas linhas normais.
# Só usa resultados já calculados nas células anteriores. Não altera nada no lake.
# ------------------------------------------------------------
versoes_final = {
    "v1 (k=5)":   (params_v1, "z_v1", 5),
    "v2 (k=3)":   (params_v2, "z_v2", 3),
    "v2 (k=3.5)": (params_v2, "z_v2", 3.5),
}
mae_normais = mae(teste_normais["preco"], teste_normais["previsto"])

colunas_resumo = {}
for nome, (p, coluna_z, k) in versoes_final.items():
    alerta = teste_todas[coluna_z].abs() > k
    fora_dos_rotulos = alerta & (teste_todas["rotulo_fraco"] == 0) & (teste_todas["rotulo_apoio"] == 0)
    r = avaliar_alertas(teste_todas.assign(z=teste_todas[coluna_z]), k)

    d = {
        "MAE linhas normais (R$)": mae_normais,
        "MAD global": p["mad_global"],
        "limite k": k,
        "alertas no teste": r["n_alertas"],
        "alertas fora dos rótulos": int(fora_dos_rotulos.sum()),
        "precision apoio (%)": r["precision_apoio"],
        "recall apoio (%)": r["recall_apoio"],
        "precision fraco (%)": r["precision_fraco"],
        "recall fraco (%)": r["recall_fraco"],
    }
    for cenario in ["÷5", "÷3", "÷2", "x2", "x3", "x5", "x8"]:
        d[f"recall {cenario} (%)"] = tabela_sint_teste.loc[cenario, nome]
    d["falso alarme em linhas normais (%)"] = tabela_sint_teste.loc["x1 (original)", nome]
    colunas_resumo[nome] = d

resumo_evolucao = pd.DataFrame(colunas_resumo)
print("=== Evolução do baseline (teste: 765 SKUs que o modelo não viu) ===")
print(resumo_evolucao.round(3).to_string())

In [0]:
# ------------------------------------------------------------
# Dois níveis de alerta (proposta aprovada pelo Geovany)
# Classifica cada linha em "forte" (|z| acima de 5), "revisão" (|z| entre 3 e 5)
# ou "normal", usando o z da versão 2 (MAD fora do treino).
# Mostra a contagem no teste, o cruzamento com os rótulos reais e a % de linhas
# em cada nível nos erros sintéticos.
# Só lê os dados já carregados. Não altera nada no lake.
# ------------------------------------------------------------
LIMITE_FORTE = 5
LIMITE_REVISAO = K_V2   # 3, escolhido no treino (célula 9)
ORDEM = ["forte", "revisão", "normal"]

def classificar_nivel(z):
    """Nível do alerta a partir do z: 'forte', 'revisão' ou 'normal'.

    z: Series com o z de cada linha.
    """
    z_abs = z.abs()
    nivel = np.select([z_abs > LIMITE_FORTE, z_abs > LIMITE_REVISAO],
                      ["forte", "revisão"], default="normal")
    return pd.Series(nivel, index=z.index)

teste_todas["nivel_alerta"] = classificar_nivel(teste_todas["z_v2"])

# 1) Contagem no teste e cruzamento com os rótulos reais.
def contar(df):
    return df["nivel_alerta"].value_counts().reindex(ORDEM, fill_value=0)

sem_rotulo = (teste_todas["rotulo_fraco"] == 0) & (teste_todas["rotulo_apoio"] == 0)
tabela_nivel = pd.DataFrame({
    "linhas": contar(teste_todas),
    "no rótulo fraco": contar(teste_todas[teste_todas["rotulo_fraco"] == 1]),
    "no rótulo de apoio": contar(teste_todas[teste_todas["rotulo_apoio"] == 1]),
    "sem rótulo": contar(teste_todas[sem_rotulo]),
})
print("=== Teste (", len(teste_todas), "linhas): alertas por nível ===")
print(tabela_nivel.to_string())

# Conferência: forte + revisão tem que bater com o alerta da v2 com k = 3.
total_alertas = int((teste_todas["nivel_alerta"] != "normal").sum())
print("\nForte + revisão:", total_alertas,
      "| alertas da v2 com k = 3:", int((teste_todas["z_v2"].abs() > K_V2).sum()))

# 2) Erros sintéticos do teste: % de linhas em cada nível.
linhas_sint = []
for cenario, df_c in sinteticos.items():
    nivel = classificar_nivel(calcular_z(df_c, params_v2))
    linhas_sint.append({
        "cenário": cenario,
        "forte (%)": (nivel == "forte").mean() * 100,
        "revisão (%)": (nivel == "revisão").mean() * 100,
        "total em alerta (%)": (nivel != "normal").mean() * 100,
    })
tabela_nivel_sint = pd.DataFrame(linhas_sint).set_index("cenário")
print("\n=== Teste, erros sintéticos: % de linhas por nível de alerta ===")
print("(erros: recall | x1 (original): falso alarme)\n")
print(tabela_nivel_sint.round(1).to_string())

%md
## Registro de evoluções do baseline

Cada linha mostra o resultado antes, o que foi alterado, o resultado depois e o ganho. Tudo medido no teste (765 SKUs que o modelo não viu), com erros sintéticos nas linhas normais.

| # | O que foi alterado | Antes | Depois | Ganho | Notebook |
|---|---|---|---|---|---|
| 0 | Ponto de partida (baseline do notebook 09): LightGBM com 7 colunas, 68 árvores, k = 5, MAD calculada no treino | MAE R\$ 9,45; MAD global 0,336; 1.334 alertas; recall: 8,0% em ÷5, 0,4% em ×5 e 13,0% em ×8 | Referência para todas as linhas abaixo | Não se aplica | 09 |
| 1 | Colunas do modelo: de 12 (com marca e variante) para 7 (categoria, categoria pai, unidade e 4 medidas de tamanho) | MAE R\$ 9,53 | MAE R\$ 9,45 | −R\$ 0,08 (cerca de 0,8%) | 09 |
| 2 | Limite do alerta: k = 5 para k = 3 (escolhido no treino, com falso alarme de no máximo 0,05%) | Recall: 8,0% em ÷5, 0,4% em ×5 e 13,0% em ×8 | Recall: 57,6% em ÷5, 57,9% em ×5 e 81,6% em ×8 | +49,6, +57,5 e +68,6 pontos percentuais; rótulos reais seguem em 100%, com 9 alertas a mais em 58.546 linhas | 10 |
| 3 | MAD calculada com previsões fora do fold (em vez das linhas que o modelo já viu) | MAD global 0,336 | MAD global 0,338 | Sem ganho relevante; mantida por ser o cálculo mais defensável | 10 |
| 4 | Dois níveis de alerta: forte (\|z\| acima de 5) e revisão (\|z\| entre 3 e 5) | Um só nível | Forte: 1.334 linhas; revisão: 9 linhas | O recall total não muda (57,6% em ÷5); passa a separar o forte (7,7%) do que vai para revisão (49,9%) | 10 |
| 5 | Segunda técnica: Isolation Forest no lugar do LightGBM, com o mesmo limite de falso alarme no treino | Isolation Forest: 22,1% em ÷5, 6,7% em ×5 e 13,0% em ×8; 17 falsos alarmes | LightGBM: 57,6% em ÷5, 57,9% em ×5 e 81,6% em ×8; 9 falsos alarmes | LightGBM pega +35,5, +51,2 e +68,6 pontos percentuais a mais | 11 |

**Ponto de atenção:** uma regra simples (mediana por subcategoria e tamanho) fica no mesmo nível do LightGBM (62,4% em ÷5, 61,4% em ×5 e 84,1% em ×8, com 13 falsos alarmes). O modelo ainda não mostrou vantagem sobre essa referência; é isso que as próximas evoluções precisam mudar.

%md
## Conclusões da evolução do baseline

**O que foi feito:** partindo do baseline do notebook 09 (LightGBM, 7 colunas, 68 árvores, MAE de R\$ 9,45 nas linhas normais do teste), medimos o alerta em erros moderados injetados de forma sintética, corrigimos o cálculo da MAD com previsões fora do fold, escolhemos o limite k sem usar o teste e implementamos dois níveis de alerta.

1. **Erros sintéticos:** multiplicamos o preço das 57.212 linhas normais do teste por fatores de ÷5 a ×8. Como todas as linhas passam a ser erros de tamanho conhecido, a % de linhas em alerta é o recall; no cenário original (×1) é o falso alarme.

2. **O baseline (k = 5) pega pouco erro moderado:** recall de 8,0% em ÷5, 0,2% em ÷3, 0,4% em ×5 e 13,0% em ×8, e 0% em ÷2, ×2 e ×3. Ele cobre os erros extremos dos rótulos (razão preço real / esperado menor ou igual a 0 ou maior ou igual a 14), mas deixa passar o que fica abaixo disso.

3. **MAD fora do treino:** treinamos o modelo base 4 vezes, cada uma deixando um fold de fora, e recalculamos a MAD com essas previsões. A MAD global foi de 0,336 para 0,338 (diferença de até 1,6% por categoria raiz) e o MAE no treino de R\$ 9,20 para R\$ 9,23. O otimismo da MAD do baseline era desprezível. Mantivemos a versão 2 por ser o cálculo mais defensável, mas ela não é a fonte do ganho.

4. **Escolha do k:** com erros sintéticos nas linhas do treino (previsões fora do fold), definimos o critério antes de ver os números: o menor k com falso alarme de no máximo 0,05% nas linhas normais. Resultado: k = 3 (0,032%); k = 2,5 deu 0,43% e k = 2 deu 4,8%.

5. **Resultado no teste (765 SKUs que o modelo não viu), baseline contra k = 3:** recall de 8,0% para 57,6% em ÷5; de 0,2% para 25,9% em ÷3; de 0% para 9,9% em ÷2; de 0% para 14,3% em ×3; de 0,4% para 57,9% em ×5; de 13,0% para 81,6% em ×8. Contra os rótulos reais: 1.343 alertas (9 a mais que no baseline), recall de 100% nos dois rótulos e precision de 99,3% contra o rótulo de apoio.

6. **Os 9 alertas extras:** todos são de preço abaixo do esperado (22% a 31% do esperado e da mediana do próprio SKU), com |z| entre 3,01 e 3,13 (no limite), e dois SKUs aparecem duas vezes com preços parecidos. São plausíveis, mas não são registros claramente anômalos. Só o negócio pode dizer se são promoção ou erro.

7. **Dois níveis de alerta (aprovados pelo Geovany e implementados na célula 14):** alerta forte (|z| acima de 5), com os mesmos 1.334 alertas do baseline, 100% dos dois rótulos e nenhum alerta sem rótulo; e alerta de revisão (|z| entre 3 e 5), com 9 linhas a mais em 58.546 (0,015%). Nos erros sintéticos, o forte pega 7,7% dos ÷5 e 12,0% dos ×8, e a revisão traz mais 49,9% e 69,6%: o ganho nos erros moderados está quase todo na revisão. O k = 3,5 continua como alternativa conservadora, com 0 alertas fora dos rótulos e recall de 70,3% em ×8 e 39,3% em ×5.

8. **Limitações:** erros de 2 vezes continuam sem ser detectados (0,2% em ×2), porque o ruído natural de cerca de 35% por observação os esconde. Os erros sintéticos não substituem rótulos reais de erros moderados. Os falsos alarmes são um teto, pois as linhas normais podem conter erros que nenhuma regra marcou. O MAE não mudou (R\$ 9,45), porque o modelo é o mesmo: a evolução está no alerta.

9. **Próximos passos:** levar os alertas de revisão ao negócio para saber se são erro ou promoção; calcular a MAD por subcategoria; ajustar hiperparâmetros e testar novas colunas e limites por categoria, para o modelo superar a regra simples de mediana por subcategoria e tamanho (notebook 11).